# Week 9 Activity: resource tagging based cost governance

**Dataset:** `data/cloudmart_multi_account.csv`, the CloudMart Inc. resource tagging dataset
(one row per cloud resource across six linked AWS accounts, one per department).

This notebook works through Task Sets 1 to 5. The same analysis runs live, with filters and an
editable remediation table, in the Streamlit dashboard (`app.py`).

In [1]:
import csv
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

RAW_PATH = "data/cloudmart_multi_account.csv"
TAG_FIELDS = ["Department", "Project", "Environment", "Owner", "CostCenter", "CreatedBy"]
COLORS = {"Tagged": "#2a78d6", "Untagged": "#eb6834"}   # same two colours in every chart
money = lambda v: f"${v:,.0f}"

## Task Set 1: Data exploration

### 1.1 Load the dataset and show the first 5 rows

A plain `pd.read_csv()` does not work on this file: every line is wrapped in one pair of
quotes, so pandas reads each row as a single text value.

In [2]:
naive = pd.read_csv(RAW_PATH)
print("Shape with a plain read_csv:", naive.shape)
naive.head(2)

Shape with a plain read_csv: (72, 1)


,"AccountID,ResourceID,Service,Region,Department,Project,Environment,Owner,CostCenter,CreatedBy,MonthlyCostUSD,Tagged"
0,"1001,i-001,EC2,us-east-1,Marketing,CampaignApp..."
1,"1001,i-002,EC2,us-east-1,Marketing,CampaignApp..."


So the file is read line by line: the outer quotes are removed and each line is split on commas.
Two resources (`ebs-002`, `ec2-004`) are also **short of two fields**
(`...,us-west-1,,,,Manual,45,No`). The last three values on every row are always
`CreatedBy`, `MonthlyCostUSD` and `Tagged`, so those stay at the end and the missing tag
fields are left blank. Otherwise pandas would shift `Manual` into `Owner` and `45` into `CostCenter`.

In [3]:
with open(RAW_PATH, encoding="utf-8-sig") as f:
    lines = [line.strip() for line in f if line.strip()]

unquote = lambda line: line[1:-1] if line.startswith('"') and line.endswith('"') else line
header = next(csv.reader([unquote(lines[0])]))
rows = []
for line in lines[1:]:
    vals = next(csv.reader([unquote(line)]))
    if len(vals) < len(header):                       # short row: re-align it
        lead, tail = header.index("Department"), 3    # CreatedBy, MonthlyCostUSD, Tagged
        middle = vals[lead:-tail]
        vals = vals[:lead] + middle + [""] * (len(header) - lead - tail - len(middle)) + vals[-tail:]
    rows.append(vals)

df_raw = pd.DataFrame(rows, columns=header).replace("", pd.NA)
df_raw["MonthlyCostUSD"] = df_raw["MonthlyCostUSD"].astype(float)
print("Rows read:", len(df_raw))
print("Exact duplicate rows:", df_raw.duplicated().sum())
print("Unique ResourceIDs:", df_raw["ResourceID"].nunique())

Rows read: 72
Exact duplicate rows: 36
Unique ResourceIDs: 36


The file holds **72 rows but only 36 resources**: rows 38 to 73 repeat rows 2 to 37 exactly.
Counting them twice would double every cost, so the duplicates are removed. (Every percentage
below comes out the same either way; only the dollar totals would double.)

The cleaned data is saved as **`data/original.csv`**, the "before" dataset.

In [4]:
df = df_raw.drop_duplicates().reset_index(drop=True)
df.to_csv("data/original.csv", index=False)
df.head(5)

,AccountID,ResourceID,Service,Region,Department,Project,Environment,Owner,CostCenter,CreatedBy,MonthlyCostUSD,Tagged
0,1001,i-001,EC2,us-east-1,Marketing,CampaignApp,Prod,j.smith@cloudmart.com,CC101,Terraform,120.0,Yes
1,1001,i-002,EC2,us-east-1,Marketing,CampaignApp,Dev,NaN,CC101,Terraform,80.0,No
2,1001,s3-001,S3,us-east-1,Marketing,AdsAPI,Prod,j.smith@cloudmart.com,CC101,Jenkins,60.0,Yes
3,1001,s3-002,S3,us-east-1,Marketing,AdsAPI,Dev,NaN,CC101,Manual,25.0,No
4,1001,api-001,APIGateway,us-east-2,Marketing,AdsAPI,Prod,j.smith@cloudmart.com,CC101,Jenkins,70.0,Yes


### 1.2 Check for missing values

In [5]:
df.isnull().sum()

AccountID          0
ResourceID         0
Service            0
Region             0
Department         2
Project            2
Environment        2
Owner             19
CostCenter         2
CreatedBy          0
MonthlyCostUSD     0
Tagged             0
dtype: int64

### 1.3 Which columns have the most missing values?

In [6]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
pd.DataFrame({"missing": missing, "% of resources": (missing / len(df) * 100).round(1)})

,missing,% of resources
Owner,19,52.8
Department,2,5.6
Project,2,5.6
Environment,2,5.6
CostCenter,2,5.6


**Answer:** `Owner` is missing on **19 of 36 resources (52.8%)**. `Department`, `Project`,
`Environment` and `CostCenter` are each missing on 2 resources: `ebs-002` and `ec2-004`, which
have no tags at all apart from `CreatedBy = Manual`.

### 1.4 Count total resources, tagged vs untagged

In [7]:
print("Total resources:", len(df))
df["Tagged"].value_counts()

Total resources: 36


Tagged
No     19
Yes    17
Name: count, dtype: int64

### 1.5 What percentage of resources are untagged?

In [8]:
untagged_pct = (df["Tagged"] == "No").sum() / len(df) * 100
print(f"Untagged resources: {untagged_pct:.1f}%")

Untagged resources: 52.8%


**Answer:** 36 resources: 17 tagged and 19 untagged, so **52.8% are untagged**.

## Task Set 2: Cost visibility

### 2.1 Total cost of tagged vs untagged resources

In [9]:
cost_by_tag = df.groupby("Tagged")["MonthlyCostUSD"].sum()
total_cost = df["MonthlyCostUSD"].sum()
print("Total monthly cost:", money(total_cost))
cost_by_tag

Total monthly cost: $3,710


Tagged
No     1370.0
Yes    2340.0
Name: MonthlyCostUSD, dtype: float64

### 2.2 Percentage of total cost that is untagged

In [10]:
untagged_cost = cost_by_tag.get("No", 0)
print(f"Untagged cost: {money(untagged_cost)} of {money(total_cost)} = {untagged_cost / total_cost * 100:.1f}%")

Untagged cost: $1,370 of $3,710 = 36.9%


**Answer:** CloudMart spends **\$3,710 a month**. Tagged resources cost \$2,340 and untagged
resources **\$1,370, which is 36.9% of the bill** that cannot be fully allocated.

### 2.3 Which department has the most untagged cost?

In [11]:
dept = (df.assign(Department=df["Department"].fillna("(missing)"))
          .pivot_table(index="Department", columns="Tagged", values="MonthlyCostUSD",
                       aggfunc="sum", fill_value=0))
dept["Total"] = dept.sum(axis=1)
dept["Untagged %"] = (dept["No"] / dept["Total"] * 100).round(1)
dept.sort_values("No", ascending=False)

Tagged,No,Yes,Total,Untagged %
Department,,,,
(missing),245.0,0.0,245.0,100.0
Sales,240.0,720.0,960.0,25.0
Analytics,210.0,400.0,610.0,34.4
Finance,200.0,310.0,510.0,39.2
HR,170.0,335.0,505.0,33.7
DevOps,165.0,275.0,440.0,37.5
Marketing,140.0,300.0,440.0,31.8


**Answer:** Of the named departments, **Sales has the most untagged cost (\$240 a month)**.
A further **\$245 has no department at all** (`ebs-002` and `ec2-004`), so nobody would be
charged for it. Their AccountIDs show where they belong: `ec2-004` (\$200) is in Sales' account
1002 and `ebs-002` (\$45) in Marketing's account 1001, which puts **Sales' real untagged cost at \$440**.
Every department has untagged spend; Marketing has the least (\$140, or \$185 with `ebs-002`).

### 2.4 Which project consumes the most cost overall?

In [12]:
df.groupby("Project", dropna=False)["MonthlyCostUSD"].sum().sort_values(ascending=False)

Project
CRMTool          960.0
LedgerDB         355.0
DataLake         270.0
CI-CD            270.0
CampaignApp      250.0
PayrollApp       245.0
NaN              245.0
DataWarehouse    230.0
AdsAPI           190.0
RecruitPortal    180.0
InfraMonitor     170.0
BudgetTracker    155.0
Insights         110.0
EmployeeDocs      80.0
Name: MonthlyCostUSD, dtype: float64

**Answer:** **CRMTool (Sales) at \$960 a month**, about 26% of all spend. LedgerDB (Finance, \$355)
is second. \$245 has no project tag.

### 2.5 Compare Prod vs Dev in cost and tagging quality

In [13]:
env = (df.assign(Environment=df["Environment"].fillna("(missing)"))
         .groupby(["Environment", "Tagged"])["MonthlyCostUSD"].agg(["count", "sum"])
         .unstack(fill_value=0))
env

count         sum        
Tagged         No Yes      No     Yes
Environment                          
(missing)       2   0   245.0     0.0
Dev            14   0  1000.0     0.0
Prod            0  17     0.0  2340.0
Test            3   0   125.0     0.0

**Answer:** The split is clean and telling:

| Environment | Resources | Monthly cost | Tagged |
|---|---|---|---|
| Prod | 17 | \$2,340 | 17 of 17 (100%) |
| Dev | 14 | \$1,000 | 0 of 14 (0%) |
| Test | 3 | \$125 | 0 of 3 (0%) |
| (missing) | 2 | \$245 | 0 of 2 |

Prod costs more than twice what Dev does, and is fully tagged. **Every non-production resource
is untagged**, and the pattern lines up with how they were created: 18 of the 19 untagged
resources were created **Manually**, while 17 of the 18 resources created by Terraform,
Jenkins or CloudFormation are tagged.

In [14]:
pd.crosstab(df["CreatedBy"], df["Tagged"], margins=True)

Tagged,No,Yes,All
CreatedBy,,,
CloudFormation,0,2,2
Jenkins,0,6,6
Manual,18,0,18
Terraform,1,9,10
All,19,17,36


## Task Set 3: Tagging compliance

### 3.1 Tag Completeness Score per resource

The score counts how many of the six tag fields are filled in
(`Department`, `Project`, `Environment`, `Owner`, `CostCenter`, `CreatedBy`), from 0 to 6.

In [15]:
df["TagCompleteness"] = df[TAG_FIELDS].notna().sum(axis=1)
df["TagCompleteness%"] = (df["TagCompleteness"] / len(TAG_FIELDS) * 100).round(0)
df["TagCompleteness"].value_counts().sort_index()

TagCompleteness
1     2
5    17
6    17
Name: count, dtype: int64

17 resources score 6/6 (fully tagged), 17 score 5/6 (only `Owner` missing) and
2 score 1/6 (only `CreatedBy` present).

### 3.2 Top 5 resources with the lowest completeness scores

In [16]:
lowest = df.sort_values(["TagCompleteness", "MonthlyCostUSD"], ascending=[True, False])
lowest[["ResourceID", "Service", "AccountID", "TagCompleteness", "MonthlyCostUSD"] + TAG_FIELDS].head(5)

,ResourceID,Service,AccountID,TagCompleteness,MonthlyCostUSD,Department,Project,Environment,Owner,CostCenter,CreatedBy
13,ec2-004,EC2,1002,1,200.0,NaN,NaN,NaN,NaN,NaN,Manual
7,ebs-002,EBS,1001,1,45.0,NaN,NaN,NaN,NaN,NaN,Manual
9,rds-002,RDS,1002,5,150.0,Sales,CRMTool,Dev,NaN,CC102,Manual
28,rds-004,RDS,1005,5,110.0,Finance,LedgerDB,Dev,NaN,CC105,Manual
11,cf-002,CloudFront,1002,5,90.0,Sales,CRMTool,Dev,NaN,CC102,Manual


Ties are broken by cost (most expensive first), since that is where missing tags hide the most money.
`ec2-004` (\$200) and `ebs-002` (\$45) are the worst at 1/6. The next three are 5/6 resources
(`rds-002` \$150, `rds-004` \$110, and `cf-002` \$90, which ties on cost with `s3-004` and `ecs-002`).

### 3.3 Most frequently missing tag fields

In [17]:
df[TAG_FIELDS].isnull().sum().sort_values(ascending=False)

Owner          19
Department      2
Project         2
Environment     2
CostCenter      2
CreatedBy       0
dtype: int64

**Answer:** `Owner` (19), then `Department`, `Project`, `Environment` and `CostCenter` (2 each). `CreatedBy` is never missing.

### 3.4 All untagged resources and their costs

In [18]:
untagged = df[df["Tagged"] == "No"].sort_values("MonthlyCostUSD", ascending=False)
print(len(untagged), "untagged resources, total", money(untagged["MonthlyCostUSD"].sum()), "a month")
untagged[["ResourceID", "Service", "Region", "Department", "Project", "Environment", "Owner", "CreatedBy", "MonthlyCostUSD"]]

19 untagged resources, total $1,370 a month


,ResourceID,Service,Region,Department,Project,Environment,Owner,CreatedBy,MonthlyCostUSD
13,ec2-004,EC2,us-east-2,NaN,NaN,NaN,NaN,Manual,200.0
9,rds-002,RDS,us-west-2,Sales,CRMTool,Dev,NaN,Manual,150.0
28,rds-004,RDS,us-west-1,Finance,LedgerDB,Dev,NaN,Manual,110.0
21,ecs-002,ECS,us-west-1,DevOps,CI-CD,Dev,NaN,Manual,90.0
17,s3-004,S3,us-east-1,Analytics,DataLake,Dev,NaN,Manual,90.0
11,cf-002,CloudFront,us-east-1,Sales,CRMTool,Dev,NaN,Manual,90.0
31,ec2-006,EC2,us-east-1,HR,PayrollApp,Dev,NaN,Manual,85.0
1,i-002,EC2,us-east-1,Marketing,CampaignApp,Dev,NaN,Terraform,80.0
19,ddb-002,DynamoDB,us-west-2,Analytics,DataWarehouse,Dev,NaN,Manual,80.0
35,ec2-010,EC2,us-east-2,HR,RecruitPortal,Test,NaN,Manual,60.0


### 3.5 Export the untagged resources to CSV

In [19]:
untagged.drop(columns=["TagCompleteness", "TagCompleteness%"]).to_csv("data/untagged.csv", index=False)
print("Saved data/untagged.csv")

Saved data/untagged.csv


## Task Set 4: Visualization

These are the same charts the dashboard shows (with filters there). Tagged is always blue and
untagged always orange.

In [20]:
df["Status"] = df["Tagged"].map({"Yes": "Tagged", "No": "Untagged"})

def style(fig, **kw):
    fig.update_layout(template="plotly_white", font_family="Segoe UI, Arial", barcornerradius=4,
                      legend_title_text="", margin=dict(l=10, r=10, t=60, b=10), **kw)
    return fig

### 4.1 Pie chart: tagged vs untagged resources

In [21]:
counts = df["Status"].value_counts().reindex(["Tagged", "Untagged"]).reset_index()
fig = px.pie(counts, names="Status", values="count", hole=0.5, color="Status",
             color_discrete_map=COLORS, title="Resources by tagging status")
fig.update_traces(textinfo="label+value+percent", sort=False,
                  marker_line_color="white", marker_line_width=2)
style(fig).show()

### 4.2 Bar chart: cost per department by tagging status

In [22]:
by_dept = (df.assign(Department=df["Department"].fillna("(missing)"))
             .groupby(["Department", "Status"], as_index=False)["MonthlyCostUSD"].sum())
dept_order = (df.groupby("Department")["MonthlyCostUSD"].sum()
                .sort_values(ascending=False).index.tolist() + ["(missing)"])
fig = px.bar(by_dept, x="Department", y="MonthlyCostUSD", color="Status", barmode="group",
             color_discrete_map=COLORS, title="Monthly cost per department by tagging status",
             category_orders={"Department": dept_order, "Status": ["Tagged", "Untagged"]},
             labels={"MonthlyCostUSD": "Monthly cost (USD)"})
style(fig, bargap=0.25, bargroupgap=0.08).show()

### 4.3 Horizontal bar chart: total cost per service

In [23]:
by_svc = df.groupby(["Service", "Status"], as_index=False)["MonthlyCostUSD"].sum()
order = df.groupby("Service")["MonthlyCostUSD"].sum().sort_values().index.tolist()
fig = px.bar(by_svc, y="Service", x="MonthlyCostUSD", color="Status", orientation="h",
             color_discrete_map=COLORS, category_orders={"Service": order[::-1]},
             title="Total monthly cost per service", labels={"MonthlyCostUSD": "Monthly cost (USD)"})
fig.update_traces(marker_line_color="white", marker_line_width=1)
style(fig).show()

### 4.4 Cost by environment

In [24]:
by_env = (df.assign(Environment=df["Environment"].fillna("(missing)"))
            .groupby(["Environment", "Status"], as_index=False)["MonthlyCostUSD"].sum())
fig = px.bar(by_env, x="Environment", y="MonthlyCostUSD", color="Status",
             color_discrete_map=COLORS, title="Monthly cost by environment",
             category_orders={"Environment": ["Prod", "Dev", "Test", "(missing)"], "Status": ["Tagged", "Untagged"]},
             labels={"MonthlyCostUSD": "Monthly cost (USD)"})
style(fig, bargap=0.35).show()

### 4.5 Interactive filters

The filters are in the Streamlit dashboard: `st.multiselect()` in the sidebar for **Service,
Region, Department and Environment**, and every chart and table redraws for the selection.
The filter itself is a plain pandas mask, shown here for EC2 in us-east-1:

In [25]:
def apply_filters(data, service=None, region=None, department=None):
    mask = pd.Series(True, index=data.index)
    if service:    mask &= data["Service"].isin(service)
    if region:     mask &= data["Region"].isin(region)
    if department: mask &= data["Department"].fillna("(missing)").isin(department)
    return data[mask]

apply_filters(df, service=["EC2"], region=["us-east-1"])[["ResourceID", "Department", "Environment", "Tagged", "MonthlyCostUSD"]]

,ResourceID,Department,Environment,Tagged,MonthlyCostUSD
0,i-001,Marketing,Prod,Yes,120.0
1,i-002,Marketing,Dev,No,80.0
30,ec2-005,HR,Prod,Yes,160.0
31,ec2-006,HR,Dev,No,85.0


## Task Set 5: Tag remediation workflow

### 5.1 and 5.2 Fill the missing tags

In the dashboard's **Remediation** tab the untagged resources sit in an `st.data_editor()` table
where Department, Project, Environment, Owner and CostCenter can be typed or picked. A
**Fill suggested tags** button pre-fills them from these rules, which use only what the tagged
resources already show:

1. **Department and CostCenter from the AccountID.** Each linked account belongs to one
   department (1001 Marketing/CC101, 1002 Sales/CC102, ... 1006 HR/CC106).
2. **Owner from the Department.** Each department has one named owner on its tagged resources
   (Marketing j.smith, Sales a.lee, Analytics k.patel, DevOps m.tan, Finance l.chen, HR t.khan).
3. **Project** for the two bare resources from the tagged resource in the same account, service
   and region (`ebs-002` next to `ebs-001` CampaignApp, `ec2-004` next to `ec2-003` CRMTool).
4. **Environment** for those two: each is the second copy of a Prod resource, following the
   dataset's `xxx-001` Prod / `xxx-002` Dev pattern, so it is set to Dev.

Rules 3 and 4 are educated guesses, so they are flagged for the owner to confirm. A resource
counts as tagged once Department, Project, Environment, Owner (an email) and CostCenter are all
filled in; that rule gives exactly the dataset's own `Tagged` column on the original data.

In [26]:
REQUIRED = ["Department", "Project", "Environment", "Owner", "CostCenter"]
before = df[header].copy()
after = before.copy()
tagged_rows = before[before["Tagged"] == "Yes"]
mode_map = lambda data, k, v: data[[k, v]].dropna().groupby(k)[v].agg(lambda s: s.mode().iloc[0]).to_dict()
acct_dept, acct_cc = mode_map(before, "AccountID", "Department"), mode_map(before, "AccountID", "CostCenter")
dept_owner = mode_map(tagged_rows, "Department", "Owner")
log = []

def fill(i, col, value, rule, confirm="No"):
    log.append({"ResourceID": after.at[i, "ResourceID"], "Field": col, "After": value,
                "Rule": rule, "NeedsOwnerConfirmation": confirm})
    after.at[i, col] = value

for i in after.index:
    acct = after.at[i, "AccountID"]
    if pd.isna(after.at[i, "Department"]):
        fill(i, "Department", acct_dept[acct], f"Account {acct} belongs to {acct_dept[acct]}")
    if pd.isna(after.at[i, "CostCenter"]):
        fill(i, "CostCenter", acct_cc[acct], f"Account {acct} bills to {acct_cc[acct]}")
    if pd.isna(after.at[i, "Owner"]):
        fill(i, "Owner", dept_owner[after.at[i, "Department"]], f"Owner of the {after.at[i, 'Department']} department")
    if pd.isna(after.at[i, "Project"]):
        sib = tagged_rows[(tagged_rows["AccountID"] == acct) & (tagged_rows["Service"] == after.at[i, "Service"])
                          & (tagged_rows["Region"] == after.at[i, "Region"])].iloc[0]
        fill(i, "Project", sib["Project"], f"Same account, service and region as {sib['ResourceID']}", "Yes")
    if pd.isna(after.at[i, "Environment"]):
        fill(i, "Environment", "Dev", "Second copy of a Prod resource (xxx-001 Prod / xxx-002 Dev)", "Yes")

filled = after[REQUIRED].notna().all(axis=1) & after["Owner"].str.contains("@", na=False)
after["Tagged"] = filled.map({True: "Yes", False: "No"})
print(len(log), "tag values filled on", len({r["ResourceID"] for r in log}), "resources")
pd.DataFrame(log).head(8)

27 tag values filled on 19 resources


,ResourceID,Field,After,Rule,NeedsOwnerConfirmation
0,i-002,Owner,j.smith@cloudmart.com,Owner of the Marketing department,No
1,s3-002,Owner,j.smith@cloudmart.com,Owner of the Marketing department,No
2,api-002,Owner,j.smith@cloudmart.com,Owner of the Marketing department,No
3,ebs-002,Department,Marketing,Account 1001 belongs to Marketing,No
4,ebs-002,CostCenter,CC101,Account 1001 bills to CC101,No
5,ebs-002,Owner,j.smith@cloudmart.com,Owner of the Marketing department,No
6,ebs-002,Project,CampaignApp,"Same account, service and region as ebs-001",Yes
7,ebs-002,Environment,Dev,Second copy of a Prod resource (xxx-001 Prod /...,Yes


### 5.3 Save the updated dataset

In [27]:
after.to_csv("data/remediated.csv", index=False)
pd.DataFrame(log).to_csv("data/remediation_log.csv", index=False)
print("Saved data/remediated.csv and data/remediation_log.csv")

Saved data/remediated.csv and data/remediation_log.csv


In the dashboard the same file comes from the `st.download_button()` under the editor.

### 5.4 Cost visibility before and after remediation

In [28]:
def metrics(d):
    unt = d["Tagged"] == "No"
    total = d["MonthlyCostUSD"].sum()
    return {"Untagged resources": int(unt.sum()),
            "Untagged resources %": round(unt.mean() * 100, 1),
            "Untagged cost (USD/month)": d.loc[unt, "MonthlyCostUSD"].sum(),
            "Untagged cost %": round(d.loc[unt, "MonthlyCostUSD"].sum() / total * 100, 1),
            "Cost visibility %": round((1 - d.loc[unt, "MonthlyCostUSD"].sum() / total) * 100, 1),
            "Cost with no Department (USD)": d.loc[d["Department"].isna(), "MonthlyCostUSD"].sum(),
            "Resources with no Owner": int(d["Owner"].isna().sum())}

pd.DataFrame({"Before": metrics(before), "After": metrics(after)})

,Before,After
Untagged resources,19.0,0.0
Untagged resources %,52.8,0.0
Untagged cost (USD/month),1370.0,0.0
Untagged cost %,36.9,0.0
Cost visibility %,63.1,100.0
Cost with no Department (USD),245.0,0.0
Resources with no Owner,19.0,0.0


In [29]:
dept_cmp = pd.DataFrame({
    "Allocated before (tagged only)": before[before["Tagged"] == "Yes"].groupby("Department")["MonthlyCostUSD"].sum(),
    "Allocated after": after.groupby("Department")["MonthlyCostUSD"].sum(),
}).fillna(0)
dept_cmp["Newly visible"] = dept_cmp["Allocated after"] - dept_cmp["Allocated before (tagged only)"]
dept_cmp.loc["Total"] = dept_cmp.sum()
dept_cmp.sort_values("Allocated after", ascending=False)

,Allocated before (tagged only),Allocated after,Newly visible
Department,,,
Total,2340.0,3710.0,1370.0
Sales,720.0,1160.0,440.0
Analytics,400.0,610.0,210.0
Finance,310.0,510.0,200.0
HR,335.0,505.0,170.0
Marketing,300.0,485.0,185.0
DevOps,275.0,440.0,165.0


**Answer:** Before remediation only **\$2,340 of \$3,710 (63.1%)** could be charged back to a
department with a named owner. After filling 27 tag values on the 19 untagged resources,
**all \$3,710 (100%)** is allocated and every resource has an owner. Sales' allocated cost rises
the most, from \$720 to \$1,160 (+\$440), because it owned the costliest untagged resources
(`ec2-004` \$200, `rds-002` \$150, `cf-002` \$90).

### 5.5 Reflection: how better tagging affects accountability and reporting

Before remediation more than a third of CloudMart's bill (\$1,370 a month, about \$16,400 a year)
was spend that nobody could be asked about. Owner was the tag most often missing, so even
where a department was known there was no person to send a cost alert to, and \$245 could not be
placed in any department's showback at all. The pattern is clear: production resources built
with Terraform, Jenkins or CloudFormation were tagged, while Dev and Test resources created by hand
were not. With the gaps filled, every dollar lands on a department, project, cost center and
owner, so chargeback/showback reports add up to the full bill, each team can see its own Dev and
Test spend (and decide whether to shut idle environments down after hours), and budgets and
anomaly alerts reach a real person. Remediation fixes the past; the lasting fix is to stop
untagged resources being created: required tags enforced at creation (AWS Organizations tag
policies and SCPs), default tags in Terraform, and a weekly compliance report that names the owner of
anything that slips through.

## Summary for the report

| Measure | Value |
|---|---|
| Resources (after removing 36 duplicate rows) | 36 |
| Untagged resources | 19 (52.8%) |
| Total monthly cost | \$3,710 |
| Untagged monthly cost | \$1,370 (36.9%) |
| Departments with missing tags | All six: Sales \$240, Analytics \$210, Finance \$200, HR \$170, DevOps \$165, Marketing \$140, plus \$245 with no department |
| Most missing tag | Owner (19 resources) |
| Untagged by environment | Prod 0 of 17, Dev 14 of 14, Test 3 of 3 |
| Cost visibility after remediation | 63.1% to 100% |